# アルゴリズム論Ⅱ 

## 第3回（発展編）：ニュートン法の真価と実務での最適適用事例
### 〜IRRの落とし穴を振り返り、「ニュートン法が圧倒的最適解となる条件」と他分野の代表事例を学ぶ〜

前回の「第2回：二分法での計算」では、中間値の定理を利用して確実に解を挟み込む方法を学びました。
そして第3回（基本編）では、関数の**微分（接線の傾き）**を利用して爆発的なスピードで解に収束する **ニュートン法（Newton-Raphson法）** を学び、投資採算性指標 **「内部収益率（IRR）」** の計算を通じて：
- わずか数ステップで収束する驚異的な計算速度（二次収束）
- 実務で直面する致命的な落とし穴（特異点への飛び火、多重解、傾きゼロでの発散＝Excelの `#NUM!` エラー）
を体験しました。

本ノートブック（発展編）では、IRRでの教訓を踏まえた上で、次の根本的な問いを探求します：
> **「では、どのような数学的構造を持つ問題であれば、落とし穴を恐れずにニュートン法を安心して使い、その圧倒的な計算速度の恩恵を最大限に享受できるのか？」**

コンピュータの基本演算（ハードウェア）、天体力学・宇宙工学、そして金融工学の実務現場から、**「ニュートン法こそが最適解」** として世界中で使われている3つの代表的事例を、Pythonコードと可視化グラフで実践的に学びます。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import math
import warnings
import logging
from scipy.stats import norm

# 警告・フォント検索ログの抑制
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# グラフ描画のスタイル設定
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# 日本語フォント対策（自動検出）
try:
    import japanize_matplotlib
except ImportError:
    candidate_fonts = ['Noto Sans CJK JP', 'IPAexGothic', 'IPAGothic', 'Hiragino Sans', 'Yu Gothic', 'Meiryo']
    available_fonts = {f.name for f in fm.fontManager.ttflist}
    found_font = next((f for f in candidate_fonts if f in available_fonts), None)
    if found_font:
        plt.rcParams['font.family'] = found_font

plt.rcParams['axes.unicode_minus'] = False
print("✅ 環境準備完了")

---
## 1. ニュートン法の原理と二次収束の威力（復習）

方程式 $f(x) = 0$ の解を求めるとき、現在地 $x_k$ における関数の接線を引き、その**接線が $x$ 軸と交わる点**を次の推測値 $x_{k+1}$ とする手法がニュートン法です。

点 $(x_k, f(x_k))$ における接線の方程式は：
$$ y - f(x_k) = f'(x_k) (x - x_k) $$

接線と $x$ 軸との交点（$y = 0$）を $x_{k+1}$ とおくと：
$$ 0 - f(x_k) = f'(x_k) (x_{k+1} - x_k) $$
これを $x_{k+1}$ について解くと、ニュートン法の更新式が得られます：

$$ x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)} $$

### 二次収束（Quadratic Convergence）の凄さ
真の解を $\alpha$（$f(\alpha) = 0$）、ステップ $k$ での誤差を $\epsilon_k = x_k - \alpha$ とします。
テイラー展開を適用すると、誤差の推移は近似的に次の関係に従います：

$$ \epsilon_{k+1} \approx \frac{f''(\alpha)}{2 f'(\alpha)} \epsilon_k^2 $$

- **二分法（一次収束）**: $\epsilon_{k+1} \approx \frac{1}{2} \epsilon_k$ （ステップごとに誤差が半分、1桁の精度向上に約3.3回反復）
- **ニュートン法（二次収束）**: $\epsilon_{k+1} \propto \epsilon_k^2$ （**ステップごとに正しい有効桁数が約2倍に増える！**）

例えば誤差が $10^{-2}$（2桁正確）であれば、次は $(10^{-2})^2 = 10^{-4}$（4桁）、次は $10^{-8}$（8桁）、次は $10^{-16}$（16桁：倍精度浮動小数点の物理限界）へと、わずか数ステップで爆発的に精度が向上します。

---
## 2. 題材の振り返り：IRR（内部収益率）と実務での3つの落とし穴

基本編で扱った投資採算性指標 **「内部収益率（IRR: Internal Rate of Return）」** は、初期投資額 $I_0$ と各年のキャッシュフロー $C_t$ に対し、正味現在価値（NPV）がゼロになる割引率 $r$ を求める問題でした：

$$ \text{NPV}(r) = -I_0 + \sum_{t=1}^T \frac{C_t}{(1 + r)^t} = 0 $$
$$ \text{NPV}'(r) = -\sum_{t=1}^T \frac{t \cdot C_t}{(1 + r)^{t+1}} $$

導関数が解析的に計算できるためニュートン法が使われますが、実務では次の **3つの致命的な落とし穴** に直面します：

1. **特異点 $r = -1$（年利 $-100\%$）への飛び火**:
   分母に $(1+r)$ があるため、$r \le -1$ に接線が跳ね飛ばされるとゼロ除算や複素数エラーでプログラムがクラッシュする。
2. **多重解の存在**:
   追加投資や撤退費用でキャッシュフローの符号が複数回反転すると、$\text{NPV}(r) = 0$ となる解が複数個（例: 10% と 20%）現れ、初期値によって求まる答えが変わってしまう。
3. **極値・傾きゼロ（$f'(r) \approx 0$）での大爆発**:
   関数の頂点付近を踏むと接線が水平に近くなり、$x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}$ の分母が極小になって**値が宇宙の彼方に吹き飛び、永久に収束しない**。これが Excel の `IRR` 関数で頻発する `#NUM!` エラーの正体でした。

In [ ]:
# IRRにおけるニュートン法の光と影（復習コード）

# --- 【光】素直な投資案件：わずか4回で10桁以上収束 ---
cf_normal = [-1000, 250, 320, 380, 420, 280]

def npv_normal(r):
    return sum(c / ((1.0 + r)**t) for t, c in enumerate(cf_normal))

def npv_normal_prime(r):
    return sum(-t * c / ((1.0 + r)**(t + 1)) for t, c in enumerate(cf_normal) if t > 0)

r = 0.10
print("=== 【光】素直な投資案件（初期値 r0 = 10%） ===")
for step in range(1, 6):
    r_next = r - npv_normal(r) / npv_normal_prime(r)
    print(f"Step {step}: r = {r_next * 100:.8f}% (変化量: {abs(r_next - r):.2e})")
    if abs(r_next - r) < 1e-10:
        print(f"🎉 わずか {step} 回で完全収束！求まったIRR = {r_next * 100:.4f}%\n")
        break
    r = r_next

# --- 【影】符号が2回反転する案件：傾きゼロ付近からの発散（#NUM!再現） ---
# 0年目: -100億, 1年目: +230億, 2年目: -132億（解は 10% と 20% の2つ）
cf_mine = [-100, 230, -132]

def npv_mine(r):
    return cf_mine[0] + cf_mine[1]/(1+r) + cf_mine[2]/((1+r)**2)

def npv_mine_prime(r):
    return -cf_mine[1]/((1+r)**2) - 2*cf_mine[2]/((1+r)**3)

# 頂点付近 (r = 14.78%) からスタート
r_bad = 0.1478
print("=== 【影】多重解・傾きゼロ付近からのスタート（初期値 r0 = 14.78%） ===")
for step in range(1, 6):
    slope = npv_mine_prime(r_bad)
    r_bad_next = r_bad - npv_mine(r_bad) / slope
    print(f"Step {step}: 傾き={slope:+.4e} --> 次の r = {r_bad_next * 100:+.2f}%")
    r_bad = r_bad_next
print("❌ 傾きが極小のため解が激しく発散（Excelの #NUM! エラー）\n")

---
## 3. なぜIRRでは失敗したのか？ニュートン法が「真価を発揮する」3大条件

IRRの計算式 $\text{NPV}(r)$ は、キャッシュフローの符号次第で「多項式のような複雑なうねり（山や谷）」を生み出し、特異点（$r = -1$）を持ち、単調性も凸性も保証されない関数でした。

この教訓から逆算すると、**「ニュートン法を安心して使い、その爆発的な計算スピードを100%享受できる問題」** には、明確な共通点があります。それが次の **3大条件** です：

| 条件 | 数学的意味 | ニュートン法にとってのメリット |
| :--- | :--- | :--- |
| **① 傾きがゼロにならない（単調性）** | 探索領域全体で $|f'(x)| \ge c > 0$ | 分母がゼロに近づいて値が吹き飛ぶ危険がない。<br>解が**唯一（一意）**に定まり、初期値によって解が変わる心配がない。 |
| **② 特異点のない領域・凸性（Convexity）** | $f''(x) > 0$（下に凸）または $f''(x) < 0$（上に凸） | 接線が常に関数の片側を通り、解に向かって**単調に吸い寄せられる（大域的収束性）**。どんな初期値からでも安全。 |
| **③ 導関数 $f'(x)$ が解析的に求まる** | $f'(x)$ の閉形式（計算式）が存在する | 差分近似（数値微分）が不要で、計算コストが極めて低く、丸め誤差の影響を受けない。 |

次節以降では、この3大条件を見事に満たし、**科学技術・ハードウェア・金融工学の現場で「ニュートン法こそが第一選択肢」とされている3つの事例** を見ていきます。

---
## 4. 事例①：コンピュータの基本演算（平方根と逆数の超高速計算）

現代のCPUやGPU、電卓は、平方根 $\sqrt{a}$ や割り算 $1/a$ をどのように超高速に計算しているのでしょうか？
実は、ハードウェア内部の演算回路（FPU: 浮動小数点演算器）では、**ニュートン法がそのまま組み込まれて動いています**。

### (1) 平方根の計算：バビロニア法（古代バビロニアから続くニュートン法）
正の実数 $a > 0$ に対し、その平方根 $x = \sqrt{a}$ を求めることは、方程式：
$$ f(x) = x^2 - a = 0 \quad (x > 0) $$
の解を求めることと同じです。

- 導関数：$f'(x) = 2x$
- ニュートン法の更新式：
  $$ x_{k+1} = x_k - \frac{x_k^2 - a}{2x_k} = \frac{1}{2} \left( x_k + \frac{a}{x_k} \right) $$

これは「現在値 $x_k$ と、割り算した値 $a/x_k$ の平均を取る」という極めて単純な式（古代バビロニア法）になります。

#### 💡 なぜニュートン法が最強なのか？（数学的証明）
1. **絶対に発散しない（大域的収束の保証）**：
   相加相乗平均の不等式より：
   $$ x_{k+1} = \frac{x_k + a/x_k}{2} \ge \sqrt{x_k \cdot \frac{a}{x_k}} = \sqrt{a} $$
   なんと、**どんな正の初期値 $x_0 > 0$ から始めても、たった1ステップで必ず $x_1 \ge \sqrt{a}$ の領域に入ります**！
   そして $x_k \ge \sqrt{a}$ においては $x_{k+1} \le x_k$（単調減少）が成り立つため、**決して発散・振動せず、坂を転がり落ちるように確実に解 $\sqrt{a}$ に収束します**。
2. **傾きが常に正**：$x > 0$ なら $f'(x) = 2x > 0$ であり、傾きゼロの罠は一切ありません。
3. **ステップごとに有効桁数が2倍**：倍精度浮動小数点の限界（16桁）に、初期値が適当でもわずか 4〜5 回で到達します。

---

### (2) 発展：除算回路を使わない逆数計算（Goldschmidt法 / 乗算型除算）
コンピュータの回路設計において、「割り算」回路は「掛け算」回路に比べて**面積が大きく、計算時間も何倍も遅い**という弱点があります。
そこで、方程式 $f(x) = \frac{1}{x} - a = 0$ にニュートン法を適用すると：

- $f'(x) = -\frac{1}{x^2}$
- 更新式：
  $$ x_{k+1} = x_k - \frac{1/x_k - a}{-1/x_k^2} = x_k + x_k^2 \left(\frac{1}{x_k} - a\right) = x_k (2 - a x_k) $$

なんと、**「引き算」と「掛け算」だけで逆数 $1/a$ が求まります（割り算を一切使わない！）**。
CPUは、指数部から粗い初期値 $x_0$ を作った後、このニュートン法を数回ハードウェアパイプラインで実行することで、超高速な割り算を実現しています。

In [ ]:
# 事例①：平方根（バビロニア法）と除算を使わない逆数計算の実装

# 1. 平方根の計算 (例: sqrt(612))
a = 612.0
true_sqrt = math.sqrt(a)

# あえて解 (約 24.74) から大きく離れた x0 = 100.0 からスタート
x = 100.0
sqrt_history = [x]

print(f"=== 平方根 sqrt({a}) のニュートン法（真値: {true_sqrt:.15f}） ===")
print(f"初期値: x_0 = {x:.1f}")
print("-" * 75)

for step in range(1, 8):
    x_next = 0.5 * (x + a / x)
    err = abs(x_next - true_sqrt)
    sqrt_history.append(x_next)
    print(f"Step {step}: x = {x_next:20.15f},  誤差 = {err:.3e}")
    if err < 1e-15:
        print("-" * 75)
        print(f"🎉 わずか {step} ステップで倍精度浮動小数点（約16桁）の限界に到達！\n")
        break
    x = x_next

# 2. 逆数計算 (例: 1 / 3.5 を掛け算と引き算だけで計算)
val = 3.5
true_inv = 1.0 / val
x_inv = 0.2  # 初期推測値

print(f"=== 逆数 1 / {val} のニュートン法（真値: {true_inv:.15f}） ===")
print("更新式: x_{k+1} = x_k * (2 - a * x_k)  ※割り算を一切使用しない！")
print("-" * 75)
for step in range(1, 6):
    x_inv_next = x_inv * (2.0 - val * x_inv)
    err = abs(x_inv_next - true_inv)
    print(f"Step {step}: x = {x_inv_next:20.15f},  誤差 = {err:.3e}")
    x_inv = x_inv_next
print("-" * 75)

# --- 収束精度の比較グラフ（二分法 vs ニュートン法） ---
bisection_err = []
low, high = 0.0, 100.0
for _ in range(len(sqrt_history)):
    mid = (low + high) / 2.0
    bisection_err.append(abs(mid - true_sqrt))
    if mid**2 < a:
        low = mid
    else:
        high = mid

newton_err = [abs(val - true_sqrt) for val in sqrt_history]

plt.figure(figsize=(9, 4.5))
plt.semilogy(range(len(newton_err)), [max(e, 1e-16) for e in newton_err], 'ro-', linewidth=2, markersize=7, label='ニュートン法 (平方根): 2次収束')
plt.semilogy(range(len(bisection_err)), bisection_err, 'bs--', linewidth=2, markersize=7, label='二分法: 1次収束')
plt.title(f"sqrt({a}) の探索における誤差の減衰速度比較", fontsize=14)
plt.xlabel("反復ステップ数 (Step)", fontsize=12)
plt.ylabel("真値との絶対誤差（対数スケール）", fontsize=12)
plt.ylim(1e-17, 1e2)
plt.legend(fontsize=11)
plt.show()

---
## 5. 事例②：天体力学・宇宙工学「ケプラー方程式（軌道計算）」

### 宇宙を支配する方程式
ドイツの天文学者ヨハネス・ケプラー（1571〜1630）は、惑星が太陽を焦点とする楕円軌道を描くことを発見しました（ケプラーの法則）。
現在でも、**人工衛星の管制、GPSの位置測定、NASAの探査機の軌道計算** では、ある時刻 $t$ における天体の位置を決定するために、次の **ケプラー方程式（Kepler's Equation）** を解く必要があります：

$$ M = E - e \sin E $$

- $e$: 軌道の離心率（$0 \le e < 1$ なら楕円軌道、$e=0$ なら真円）
- $M$: **平均近点角**（時間に比例して一定速度で増加する仮想的な角度。$M = \frac{2\pi}{T} t$）
- $E$: **離心近点角**（楕円軌道上の天体の実際の位置を決定する幾何学的な角度）

この方程式は代数的に解くこと（初等関数による厳密解）が数学的に不可能であり、**数値計算で解くしかありません**。

---

### 方程式と導関数
$$ f(E) = E - e \sin E - M = 0 $$
導関数は：
$$ f'(E) = 1 - e \cos E $$

ニュートン法の更新式：
$$ E_{k+1} = E_k - \frac{E_k - e \sin E_k - M}{1 - e \cos E_k} $$

### 💡 なぜケプラー方程式でニュートン法が「歴史上最も美しい成功例」なのか？
1. **導関数が常に有界かつ正（傾きが絶対にゼロにならない！）**：
   楕円軌道（$0 \le e < 1$）において、$-1 \le \cos E \le 1$ ですから、
   $$ 0 < 1 - e \le f'(E) \le 1 + e $$
   が**全領域で常に成り立ちます**！
   例えばハレー彗星のような極端な楕円軌道（$e = 0.967$）であっても、$f'(E) \ge 0.033 > 0$ であり、**接線の傾きがゼロになることは宇宙空間のどこにも存在しません**。
2. **特異点や局所極値が皆無**：
   関数は滑らかな単調増加関数であり、IRRのような特異点や局所的な山・谷が一切ありません。
3. **初期値 $E_0 = M$ とするだけで3〜4回で収束**：
   時刻パラメータである $M$ をそのまま初期値とするだけで、どんな離心率でもわずか3〜4回の反復で倍精度の限界まで一直線に収束します。

このため、17世紀にアイザック・ニュートン自身が考案して以来、現代の宇宙開発に至るまでニュートン法が絶対的な標準アルゴリズムとして君臨し続けています。

In [ ]:
# 事例②：ケプラー方程式の求解と人工衛星の軌道シミュレーション

def solve_kepler(M, e, tol=1e-14, max_iter=15):
    """ケプラー方程式 E - e*sin(E) = M をニュートン法で解く"""
    E = M  # 初期値は M を採用
    history = [E]
    for _ in range(max_iter):
        f = E - e * np.sin(E) - M
        f_prime = 1.0 - e * np.cos(E)
        E_next = E - f / f_prime
        history.append(E_next)
        if abs(E_next - E) < tol:
            break
        E = E_next
    return E, history

# 高離心率軌道（e = 0.65 の強い長楕円軌道）でテスト
e_orbit = 0.65
M_sample = np.radians(60.0)  # 平均近点角 60度

E_sol, hist = solve_kepler(M_sample, e_orbit)

print(f"=== ケプラー方程式の求解 (離心率 e = {e_orbit}, M = {np.degrees(M_sample):.1f}°) ===")
print(f"初期値 E_0 = {np.degrees(M_sample):.1f}°")
print("-" * 75)
for step, val in enumerate(hist):
    residual = abs(val - e_orbit * np.sin(val) - M_sample)
    print(f"Step {step}: E = {np.degrees(val):12.8f}°,  残差 |f(E)| = {residual:.3e}")
print("-" * 75)
print(f"🎉 わずか 4 回で誤差 10^-16 （倍精度限界）に完全収束！\n")

# --- 楕円軌道と天体位置の可視化 ---
# 1周分（M = 0 から 2pi）の時間を等間隔（12個のタイムステップ）でサンプリング
M_steps = np.linspace(0, 2*np.pi, 13)[:-1]
E_steps = [solve_kepler(m, e_orbit)[0] for m in M_steps]

# 軌道長半径 a = 1 としたときの軌道座標 (x, y)
# x = cos(E) - e,  y = sqrt(1 - e^2) * sin(E)  （原点 (0, 0) は焦点＝太陽・地球の位置）
x_pts = [np.cos(E) - e_orbit for E in E_steps]
y_pts = [np.sqrt(1 - e_orbit**2) * np.sin(E) for E in E_steps]

# 滑らかな楕円軌道ライン
E_fine = np.linspace(0, 2*np.pi, 200)
x_orbit = np.cos(E_fine) - e_orbit
y_orbit = np.sqrt(1 - e_orbit**2) * np.sin(E_fine)

plt.figure(figsize=(9, 5))
plt.plot(x_orbit, y_orbit, 'b-', linewidth=2, label='衛星・惑星の楕円軌道 (e = 0.65)')
plt.plot(0, 0, 'y*', markersize=16, markeredgecolor='orange', label='中心天体 (焦点: 太陽/地球)')
plt.scatter(x_pts, y_pts, color='crimson', s=60, zorder=5, label='等時間間隔ごとの位置（ケプラー方程式の解）')

# 近地点での加速（面積速度一定）を示す矢印
plt.annotate('近地点（速度最速）', xy=(x_pts[0], y_pts[0]), xytext=(0.5, 0.4),
             arrowprops=dict(facecolor='black', shrink=0.08, width=1, headwidth=6), fontsize=11)
plt.annotate('遠地点（速度最遅）', xy=(x_pts[6], y_pts[6]), xytext=(-1.5, 0.4),
             arrowprops=dict(facecolor='black', shrink=0.08, width=1, headwidth=6), fontsize=11)

plt.title("ケプラー方程式で求めた等時間間隔における天体の運行位置", fontsize=14)
plt.xlabel("X 座標（軌道長半径 a 単位）", fontsize=12)
plt.ylabel("Y 座標（軌道長半径 a 単位）", fontsize=12)
plt.axis('equal')
plt.legend(fontsize=10, loc='lower right')
plt.show()

---
## 6. 事例③：金融工学・クオンツ「インプライド・ボラティリティ（IV）の算出」

### IRRの失敗を克服した金融工学の標準
第2節で見たように、投資指標の「IRR」はキャッシュフローの符号反転による多重解や極値（傾きゼロ）のせいで、ニュートン法で `#NUM!` エラーが発生しました。
しかし同じ金融の世界において、**「ニュートン法を使わない理由がない」ほど完全に数学的条件を満たしている最重要指標**があります。
それがオプション取引における **「インプライド・ボラティリティ（IV: Implied Volatility）」** です。

### インプライド・ボラティリティとは？
ノーベル経済学賞を受賞した「ブラック・ショールズ・モデル（Black-Scholes Model）」では、株価 $S$、権利行使価格 $K$、満期までの期間 $T$、金利 $r$、将来の価格変動率（ボラティリティ）$\sigma$ から、コールオプションの理論価格 $C_{BS}(\sigma)$ を計算できます：

$$ C_{BS}(\sigma) = S \Phi(d_1) - K e^{-r T} \Phi(d_2) $$
$$ d_1 = \frac{\ln(S/K) + (r + \frac{1}{2}\sigma^2)T}{\sigma \sqrt{T}}, \quad d_2 = d_1 - \sigma \sqrt{T} $$
（$\Phi(x)$ は標準正規分布の累積分布関数）

しかし現実の市場では、逆に**「市場で取引されているオプション価格 $C_{\text{market}}$」** の方が画面に表示されます。
投資家は、「市場参加者は将来の変動率を何％と織り込んでこのプレミアムをつけているのか？」を知りたいため、逆問題である方程式：

$$ f(\sigma) = C_{BS}(\sigma) - C_{\text{market}} = 0 $$
を解いてボラティリティ $\sigma$ を逆算します。これがインプライド・ボラティリティ（IV）です。

---

### 💡 なぜIVの計算ではニュートン法が「金融業界標準」なのか？
1. **導関数（Vega: ベガ）が数学的に常に正（狭義単調増加！）**：
   オプション価格のボラティリティに対する感応度（微分）は金融業界で **Vega（ベガ）** と呼ばれ、驚くほど綺麗な式になります：
   $$ \nu(\sigma) = \frac{\partial C_{BS}}{\partial \sigma} = S \sqrt{T} \phi(d_1) > 0 $$
   （$\phi(x) = \frac{1}{\sqrt{2\pi}} e^{-x^2/2}$ は標準正規分布の確率密度関数）
   株価 $S>0$、期間 $T>0$ である限り、正規分布の密度 $\phi(d_1)$ は常に正なので、**Vega は全領域で常に正（$\nu > 0$）** です！
2. **多重解が絶対に存在しない（解の一意性）**：
   Vega が常に正ということは、ボラティリティ $\sigma$ が上がればオプション価格 $C_{BS}$ は**常に一方通行で上昇する（狭義単調増加）** ことを意味します。
   したがって、IRRのように「解が2つある」ような混乱は数学的に100%あり得ず、**解は地球上にただ1つしか存在しません**。
3. **極値（傾きゼロ）がない**：
   山や谷がないため、接線が水平になって宇宙の彼方に発散する危険が極めて低く、初期値 $\sigma_0 = 0.2$（20%）から始めれば、**世界中どの市場のどんな銘柄でもわずか3ステップ程度で瞬時に収束**します。

In [ ]:
# 事例③：ブラック・ショールズ方程式とインプライド・ボラティリティ（IV）の算出

def bs_call_price(S, K, T, r, sigma):
    """ブラック・ショールズ方程式によるコールオプション理論価格"""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def bs_vega(S, K, T, r, sigma):
    """導関数 Vega: dC / d(sigma) = S * sqrt(T) * phi(d1)  --> 常に正！"""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return S * np.sqrt(T) * norm.pdf(d1)

def calc_implied_volatility(C_market, S, K, T, r, sigma_init=0.5, tol=1e-8):
    """ニュートン法によるインプライド・ボラティリティ（IV）の逆算"""
    sigma = sigma_init
    history = [sigma]
    for step in range(1, 10):
        price = bs_call_price(S, K, T, r, sigma)
        diff = price - C_market
        vega = bs_vega(S, K, T, r, sigma)
        
        sigma_next = sigma - diff / vega
        history.append(sigma_next)
        if abs(sigma_next - sigma) < tol:
            break
        sigma = sigma_next
    return sigma, history

# 市場パラメータ設定
S_stock = 100.0   # 現在の株価 100ドル
K_strike = 105.0  # 権利行使価格 105ドル
T_year = 1.0      # 満期まで1年
r_rate = 0.03     # 無リスク金利 3%
true_vol = 0.25   # 市場の真のボラティリティ 25%

# 市場で観測されたオプション価格
C_market_obs = bs_call_price(S_stock, K_strike, T_year, r_rate, true_vol)

# 初期値を適当に年率 50% (sigma = 0.50) から開始
iv_sol, iv_hist = calc_implied_volatility(C_market_obs, S_stock, K_strike, T_year, r_rate, sigma_init=0.50)

print(f"=== インプライド・ボラティリティ（IV）のニュートン法逆算 ===")
print(f"市場観測オプション価格: ${C_market_obs:.4f}")
print(f"初期値: sigma_0 = {iv_hist[0] * 100:.1f}%")
print("-" * 75)
for step, s in enumerate(iv_hist):
    err = abs(s - true_vol)
    print(f"Step {step}: sigma = {s * 100:10.6f}%,  真値との誤差 = {err:.3e}")
print("-" * 75)
print(f"🎉 わずか 3 ステップで誤差 10^-16 に到達！求まった IV = {iv_sol * 100:.2f}%\n")

# --- 接線による収束過程の可視化 ---
sig_range = np.linspace(0.10, 0.60, 200)
prices = [bs_call_price(S_stock, K_strike, T_year, r_rate, s) for s in sig_range]

plt.figure(figsize=(9, 5))
plt.plot(sig_range * 100, prices, 'b-', linewidth=2.5, label='オプション価格曲線 C(σ)（狭義単調増加）')
plt.axhline(C_market_obs, color='black', linestyle='--', linewidth=1, label=f'市場観測価格 = ${C_market_obs:.2f}')

# 接線の描画（Step 0 から Step 1 へのジャンプ）
s0 = iv_hist[0]
p0 = bs_call_price(S_stock, K_strike, T_year, r_rate, s0)
v0 = bs_vega(S_stock, K_strike, T_year, r_rate, s0)
tangent_x = np.linspace(0.20, 0.55, 100)
tangent_y = p0 + v0 * (tangent_x - s0)
plt.plot(tangent_x * 100, tangent_y, 'g:', linewidth=2, label='初期値における接線（傾き Vega > 0）')

plt.plot(s0 * 100, p0, 'go', markersize=8, label=f'初期点 (σ = {s0*100:.0f}%)')
plt.plot(iv_sol * 100, C_market_obs, 'ro', markersize=9, label=f'収束解: IV = {iv_sol*100:.1f}%')

plt.title("インプライド・ボラティリティ逆算におけるニュートン法の収束挙動", fontsize=14)
plt.xlabel("ボラティリティ σ (%)", fontsize=12)
plt.ylabel("オプション価格 ($)", fontsize=12)
plt.ylim(0, max(prices) * 1.1)
plt.legend(fontsize=10)
plt.show()

---
## 7. まとめ：方程式の数値解法アルゴリズム 選択マップ

第2回の二分法、そして第3回のニュートン法（IRRの落とし穴と今回の3大成功事例）を通じて、数値解法の本質が見えてきました。
実務で方程式に直面したとき、どのアルゴリズムを選択すべきかを整理します。

### アルゴリズム特性の比較

| 手法 | 収束次数 | メリット | デメリット | 最適な適用シーン |
| :--- | :--- | :--- | :--- | :--- |
| **二分法 (Bisection)** | 1次収束 | **100% 確実に解に到達する**。<br>導関数不要、不連続でも符号反転があればOK。 | 収束が遅い（1桁に3〜4回反復必要）。<br>多重根や符号が変わらない解は不可。 | **絶対にクラッシュが許されない基幹システム**、関数がブラックボックスな場合 |
| **ニュートン法 (Newton)** | **2次収束** | **圧倒的に高速**（数回で倍精度限界）。<br>高精度な解が瞬時に得られる。 | 導関数 $f'(x)$ が必須。<br>傾きゼロや特異点で発散・振動する危険。 | **本ノートブックで扱った3事例**：<br>① 平方根・逆数計算（凸関数）<br>② ケプラー方程式（傾き常時正）<br>③ IV逆算（単調増加、導関数閉形式） |
| **割線法 (Secant)** | 1.62次収束 | ニュートン法並みに高速で、**導関数 $f'(x)$ の計算が不要**（差分で代用）。 | ニュートン法同様、初期値次第で発散するリスクがある。 | 微分が解析的に求まらないが、高速に解きたい場合 |
| **ブレント法 (Brent)** | 局所2次収束 | **「二分法の絶対の安全性」と「補間法の超高速性」を完全融合**。現代の業界標準。 | アルゴリズムの実装がやや複雑。 | **一般的な科学技術計算の第一選択肢**（`scipy.optimize.brentq` など） |

---

### 💡 実務でのアルゴリズム選定フローチャート

```text
方程式 f(x) = 0 を数値計算で解きたい！
  │
  ├─ Q1. 導関数 f'(x) が解析的に求まり、計算コストが軽いか？
  │    ├─ NO ──> 【ブレント法 (Brent)】または【二分法】
  │    │
  │    └─ YES ──> Q2 へ進む
  │
  ├─ Q2. 探索範囲内で傾きがゼロになる点 (f'(x) ≈ 0) や特異点が存在するか？
  │    ├─ YES（IRRや高次多項式など） ──> 危険！【ブレント法】または【初期値を与えたニュートン法】
  │    │
  │    └─ NO（全領域で傾きが同符号、あるいは凸関数） ──> Q3 へ進む
  │
  └─ Q3. 単調増加/減少、あるいは凸性（Convexity）が保証されているか？
       ├─ YES（平方根、ケプラー方程式、インプライド・ボラティリティなど）
       │    └──> 🏆 【ニュートン法 (Newton-Raphson)】が圧倒的最適解！
```

### 結び
アルゴリズムの優劣は、単なる「計算スピードの速さ」だけで決まるものではありません。
**「解くべき関数の数学的構造（導関数がゼロにならないか、凸性があるか、特異点はないか）」を正しく見抜き、適材適所で手法を選択すること** こそが、アルゴリズム論を実務や研究で活かすための最も重要な知恵です。